In [1]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit,ShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from benchmark_datasets import OpenMLBenchmark
load_dotenv(".env")

/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
benchamrks = OpenMLBenchmark("classification")

In [3]:
blood_dataset = pd.read_csv("datasets/blood_dataset/blood.csv")
print(blood_dataset.head())
data_size = len(blood_dataset)
print(data_size)

   Recency  Frequency  Monetary  Time  Class
0        2         50     12500    99      1
1        0         13      3250    28      1
2        1         17      4000    36      1
3        2         20      5000    45      1
4        1         24      6000    77      0
748


In [4]:
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.neural_network import MLPClassifier,MLPRegressor

In [5]:

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
data = np.array(blood_dataset.drop(columns=["Class"]),dtype=np.float32)
targets = np.array(blood_dataset["Class"],dtype=np.float32)
def evaluate_dataset_classification(data,targets,n_splits=5,n_train_examples=32):
    sss = StratifiedShuffleSplit(n_splits=n_splits,train_size=n_train_examples)
    pfn_accuracies = []
    distilled_tree_accuracies = []
    tree_accuracies = []
    for i,(train_index, test_index) in enumerate(sss.split(data, targets)):
        print(i)
        X_train = data[train_index]
        y_train = targets[train_index]
        X_test = data[test_index]
        y_test = targets[test_index]
        # scale features: TabPFN is scale-robust, but the MLP students are not
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2_5)
        clf.fit(X_train, y_train)
        # distillation target: teacher's probability of the positive class (class 1)
        y_distil = clf.predict_proba(X_train)[:,1]
        distilled_decision_tree = RandomForestRegressor()
        distilled_decision_tree.fit(X_train_scaled,y_distil)
        pfn_predictions = clf.predict(X_test)
        print("done")
        distilled_tree_predictions = (distilled_decision_tree.predict(X_test_scaled)>0.5).astype(np.int16)
        decision_tree = RandomForestClassifier()
        decision_tree.fit(X_train_scaled,y_train)
        tree_predictions = decision_tree.predict(X_test_scaled)
        #print("---------------")
        pfn_accuracies.append(accuracy_score(y_test,pfn_predictions))
        distilled_tree_accuracies.append(accuracy_score(y_test,distilled_tree_predictions))
        tree_accuracies.append(accuracy_score(y_test,tree_predictions))
        #print(f"PFN accuracy: {accuracy_score(y_test,pfn_predictions):3f}")
        #print(f"distillated tree accuracy: {accuracy_score(y_test,distilled_tree_predictions):3f}")
        #print(f"tree accuracy: {accuracy_score(y_test,tree_predictions):3f}")
    return np.mean(pfn_accuracies).item(),np.mean(distilled_tree_accuracies).item(),np.mean(tree_accuracies).item()
print(evaluate_dataset_classification(data,targets))

0
done
1
done
2
done
3
done
4
done
(0.741340782122905, 0.7483240223463687, 0.7276536312849162)


In [6]:
def estimator_factory_PFN(task:str):
    if task=="classification":
        return TabPFNClassifier()
    return TabPFNRegressor()
def estimator_factory_Forest(task:str):
    if task=="classification":
        return RandomForestClassifier()
    return RandomForestRegressor()

In [10]:
eval_PFN = benchamrks.evaluate(estimator_factory=estimator_factory_PFN,train_size=128)
eval_tree = benchamrks.evaluate(estimator_factory=estimator_factory_Forest,train_size=128)

[classification] tic-tac-toe          acc=0.983+/-0.002 auc=0.986+/-0.007
[classification] monks-problems-2     acc=1.000+/-0.000 auc=1.000+/-0.000
[classification] sonar                acc=0.800+/-0.022 auc=0.897+/-0.023
[classification] ionosphere           acc=0.936+/-0.010 auc=0.981+/-0.007
[classification] vehicle              acc=0.801+/-0.008 auc=0.951+/-0.003
[classification] wdbc                 acc=0.962+/-0.008 auc=0.993+/-0.002
[classification] diabetes             acc=0.752+/-0.010 auc=0.822+/-0.013
[classification] ilpd                 acc=0.709+/-0.014 auc=0.729+/-0.018
[classification] balance-scale        acc=0.938+/-0.004 auc=0.991+/-0.000
[classification] blood-transfusion    acc=0.771+/-0.013 auc=0.731+/-0.015
[classification] tic-tac-toe          acc=0.807+/-0.015 auc=0.880+/-0.013
[classification] monks-problems-2     acc=0.658+/-0.018 auc=0.673+/-0.021
[classification] sonar                acc=0.765+/-0.029 auc=0.882+/-0.022
[classification] ionosphere           

In [11]:
print(eval_PFN)
print(eval_tree)

                name            task  n_rows  acc_mean   acc_std  auc_mean  \
0        tic-tac-toe  classification     958  0.983133  0.001704  0.985795   
1   monks-problems-2  classification     601  1.000000  0.000000  1.000000   
2              sonar  classification     208  0.800000  0.022361  0.896669   
3         ionosphere  classification     351  0.936323  0.009987  0.980804   
4            vehicle  classification     846  0.800836  0.007679  0.950512   
5               wdbc  classification     569  0.961905  0.007507  0.993031   
6           diabetes  classification     768  0.752188  0.009763  0.821557   
7               ilpd  classification     583  0.708571  0.013802  0.728772   
8      balance-scale  classification     625  0.937626  0.004221  0.990580   
9  blood-transfusion  classification     748  0.771290  0.012919  0.730802   

    auc_std  
0  0.007160  
1  0.000000  
2  0.022510  
3  0.006631  
4  0.003176  
5  0.001753  
6  0.013092  
7  0.017815  
8  0.000437  
9

In [12]:
scores = pd.concat([eval_PFN,eval_tree.drop(columns=["name","task","n_rows"])],axis=1)

In [22]:
scores

,name,task,n_rows,acc_mean,acc_std,name,task,n_rows,acc_mean,acc_std
0,tic-tac-toe,classification,958,0.667603,0.036253,tic-tac-toe,classification,958,0.691577,0.030288
1,monks-problems-2,classification,601,0.640070,0.021507,monks-problems-2,classification,601,0.608436,0.017833
2,sonar,classification,208,0.710227,0.055902,sonar,classification,208,0.731818,0.046632
3,ionosphere,classification,351,0.845768,0.014757,ionosphere,classification,351,0.879624,0.026718
4,vehicle,classification,846,0.655283,0.065338,vehicle,classification,846,0.577887,0.069341
5,wdbc,classification,569,0.918063,0.018208,wdbc,classification,569,0.913594,0.006408
6,diabetes,classification,768,0.691576,0.038478,diabetes,classification,768,0.707065,0.032308
7,ilpd,classification,583,0.684936,0.019567,ilpd,classification,583,0.679855,0.016458
8,balance-scale,classification,625,0.819562,0.044578,balance-scale,classification,625,0.772344,0.043884
9,blood-transfusion,classification,748,0.746648,0.015161,blood-transfusion,classification,748,0.705866,0.032904


In [12]:
dataset = benchamrks.load("sonar")

In [13]:
X = dataset.X
y = dataset.y
print(evaluate_dataset_classification(X,y,n_train_examples=4))

0
done
1
done
2
done
3
done
4
done
(0.5911764705882352, 0.5901960784313725, 0.5598039215686275)


In [14]:
bank_dataset = pd.read_csv("datasets/banking_dataset/train.csv",delimiter=";")
onehotenc = OneHotEncoder()
categorical_features = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
targets = bank_dataset["y"]
data = bank_dataset.drop(columns=["y"])
categorical = np.array(onehotenc.fit_transform(data[categorical_features]).todense())
non_categorical = np.array(data.drop(columns=categorical_features))
X = np.concat([categorical,non_categorical],axis=1)
y = np.array(targets=="yes",dtype=np.int16)
print(X.shape)
print(evaluate_dataset_classification(X[:5000],y[:5000],n_train_examples=32))

(45211, 51)
0
done
1
done
2
done
3
done
4
done
(0.9697665056360709, 0.9697665056360709, 0.9698067632850241)


In [15]:
from sklearn.metrics import mean_squared_error
def evaluate_dataset_regression(data,targets,n_splits=5,n_train_examples=32):
    sss = ShuffleSplit(n_splits=n_splits,train_size=n_train_examples)
    pfn_accuracies = []
    distilled_tree_accuracies = []
    tree_accuracies = []
    for i,(train_index, test_index) in enumerate(sss.split(data, targets)):
        print(i)
        X_train = data[train_index]
        y_train = targets[train_index]
        X_test = data[test_index]
        y_test = targets[test_index]
        # scale features: TabPFN is scale-robust, but the MLP students are not
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        clf = TabPFNRegressor()
        clf.fit(X_train, y_train)
        y_distil = clf.predict(X_train)
        distilled_decision_tree = MLPRegressor(hidden_layer_sizes=(50,))#RandomForestRegressor()
        distilled_decision_tree.fit(X_train_scaled,y_distil)
        pfn_predictions = clf.predict(X_test)
        print("done")
        distilled_tree_predictions = distilled_decision_tree.predict(X_test_scaled)
        decision_tree = MLPRegressor(hidden_layer_sizes=(50,))#RandomForestRegressor()
        decision_tree.fit(X_train_scaled,y_train)
        tree_predictions = decision_tree.predict(X_test_scaled)
        #print("---------------")
        pfn_accuracies.append(mean_squared_error(y_test,pfn_predictions))
        distilled_tree_accuracies.append(mean_squared_error(y_test,distilled_tree_predictions))
        tree_accuracies.append(mean_squared_error(y_test,tree_predictions))
        #print(f"PFN accuracy: {accuracy_score(y_test,pfn_predictions):3f}")
        #print(f"distillated tree accuracy: {accuracy_score(y_test,distilled_tree_predictions):3f}")
        #print(f"tree accuracy: {accuracy_score(y_test,tree_predictions):3f}")
    return np.mean(pfn_accuracies).item(),np.mean(distilled_tree_accuracies).item(),np.mean(tree_accuracies).item()

In [16]:
california_housing = pd.read_csv("datasets/california_housing/housing.csv").dropna()
target = california_housing["median_house_value"]
data = california_housing.drop(columns=["median_house_value"])
categorical = np.array(onehotenc.fit_transform(data[["ocean_proximity"]]).todense(),dtype=np.float32)
non_categorical = np.array(data.drop(columns=["ocean_proximity"]),dtype=np.float32)
X = np.concat([categorical,non_categorical],axis=1)
y = np.array(target,dtype=np.float32)
print(X.shape)
evaluate_dataset_regression(X[:1000],y[:1000],n_splits=5,n_train_examples=32)

(20433, 13)
0


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
1


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
2


...

done
3


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
4


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


KeyboardInterrupt: 